In [3]:
import torch

In [4]:
# Intitializing tensors

tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1, 2, 3])
tensor2d = torch.tensor([[1, 2, 3], [4, 5, 6]])
tensor3d = torch.tensor([[[1, 2, 3], [4, 5, 6]], [[7, 8, 9], [10, 11, 12]]])

print(tensor0d.shape)
print(tensor1d.shape)
print(tensor2d.shape)
print(tensor3d.shape)

torch.Size([])
torch.Size([3])
torch.Size([2, 3])
torch.Size([2, 2, 3])


In [5]:
# Inspect data type

print(tensor1d.dtype)
tensor1d_float = torch.tensor([1.0, 2.0, 3.0])
print(tensor1d_float.dtype)

# Convert data types
tensor1d_float64 = tensor1d.to(torch.float64)
print(tensor1d_float64.dtype)

torch.int64
torch.float32
torch.float64


In [6]:
# reshaping tensor

print(tensor2d.shape)
tensor2d_reshape = tensor2d.reshape(3, 2)

# We note that there is a more common command for reshaping which is .view()

print(tensor2d_reshape.shape)
tensor2d_reshape2 = tensor2d.view(3, 2)
print(tensor2d_reshape2.shape)

torch.Size([2, 3])
torch.Size([3, 2])
torch.Size([3, 2])


In [7]:
# Transposing a tensor: Flipping across it is diagonal != reshape
# Example
print(tensor2d)
print(tensor2d.reshape(3, 2))
print(tensor2d.T)

tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 2],
        [3, 4],
        [5, 6]])
tensor([[1, 4],
        [2, 5],
        [3, 6]])


In [8]:
# Multiply matrices: Matmul

print(tensor2d.matmul(tensor2d.T))

tensor([[14, 32],
        [32, 77]])


In [9]:
# Computing values bia autograd

import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

u = w1 * x1 + b
y_pred = F.sigmoid(u)

Loss = F.binary_cross_entropy(y_pred, y)

grad_L_w1 = grad(Loss, w1, retain_graph=True)
grad_L_b = grad(Loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [10]:
# There is a better way to call the loss that gives the same output

Loss.backward()

print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


In [11]:
# Create a simple feeed forward network


class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = torch.nn.Sequential(
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),
            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            # Output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits


# Add a manual seed before the model initializes so the resuls are reporducible
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(num_params)

# Observe weights of the model

print(model.layers[0].weight)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)
2213
Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [12]:
# run a forwards pass
torch.manual_seed(123)
x = torch.rand((1, 50))
print(x.shape)
out = model(x)
print(out.shape)
print(out)

# To run a fowrad pass without gradients we can use context

with torch.no_grad():
    out = model(x)
    print(out)

torch.Size([1, 50])
torch.Size([1, 3])
tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)
tensor([[-0.1262,  0.1080, -0.1792]])


In [13]:
# Create some sample data

X_train = torch.tensor(
    [[-1.2, 3.1], [-0.9, 2.9], [-0.5, 2.6], [2.3, -1.1], [2.7, -1.5]]
)
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([[-0.8, 2.8], [2.6, -1.6]])
y_test = torch.tensor([0, 1])

In [14]:
# Create a datase
from torch.utils.data import Dataset


class ToyDataset(Dataset):
    def __init__(self, x, y):
        self.features = x
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]


train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

print(len(train_ds))

5


In [ ]:
# Create data loaders

from torch.utils.data import DataLoader

torch.manual_seed(123)
train_dl = DataLoader(
    train_ds, batch_size=2, shuffle=True, num_workers=0, drop_last=True
)
test_dl = DataLoader(test_ds, batch_size=2, shuffle=False, num_workers=0)

for idx, (x, y) in enumerate(train_dl):
    print(f"Batch {idx}:", x, y)

Batch 0: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])


In [ ]:
# Create a training loop

import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(2, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
num_epochs = 3
for epoch in range(num_epochs):
    for idx, (x, y) in enumerate(train_dl):
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ## Logger
        print(
            f"Epoch: {epoch + 1:03d}/{num_epochs:03d} | Batch {idx + 1:03d}/{len(train_dl):03d} | Loss: {loss:.2f}"
        )

Epoch: 001/003 | Batch 001/002 | Loss: 0.75
Epoch: 001/003 | Batch 002/002 | Loss: 0.65
Epoch: 002/003 | Batch 001/002 | Loss: 0.44
Epoch: 002/003 | Batch 002/002 | Loss: 0.13
Epoch: 003/003 | Batch 001/002 | Loss: 0.03
Epoch: 003/003 | Batch 002/002 | Loss: 0.00


In [ ]:
# Create an evaluation function


def compute_evaluation(model, data_loader):

    model = model.eval()
    correct = 0
    total = 0
    for idx, (x, y) in enumerate(data_loader):
        with torch.no_grad():
            logits = model(x)

        predictions = torch.argmax(logits, dim=1)
        correct += torch.sum(predictions == y)
        total += len(y)

    accuracy = correct / total
    return accuracy.item()


print(compute_evaluation(model, train_dl))
print(compute_evaluation(model, test_dl))


1.0
1.0


In [21]:
# Saving the model

torch.save(model.state_dict(), "a1_model.pth")

model2 = NeuralNetwork(2, 2)
model2.load_state_dict(torch.load("a1_model.pth"))
print(compute_evaluation(model2, test_dl))

1.0


In [ ]:
print(torch.cuda.is_available())  # <If a GPU is available
print(
    torch.backends.mps.is_available()
)  # This is the equivalent for MacOS working on silicon devices.

False
True


In [24]:
# Modify the training loop to run on a GPU

import torch.nn.functional as F

# Define device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

torch.manual_seed(123)
model = NeuralNetwork(2, 2)
model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
num_epochs = 3
for epoch in range(num_epochs):
    for idx, (x, y) in enumerate(train_dl):
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ## Logger
        print(
            f"Epoch: {epoch + 1:03d}/{num_epochs:03d} | Batch {idx + 1:03d}/{len(train_dl):03d} | Loss: {loss:.2f}"
        )

Epoch: 001/003 | Batch 001/002 | Loss: 0.75
Epoch: 001/003 | Batch 002/002 | Loss: 0.65
Epoch: 002/003 | Batch 001/002 | Loss: 0.44
Epoch: 002/003 | Batch 002/002 | Loss: 0.13
Epoch: 003/003 | Batch 001/002 | Loss: 0.03
Epoch: 003/003 | Batch 002/002 | Loss: 0.00
